# 05. 실시간 엔드포인트 배포

## 이 노트북에서 배우는 것
- 최적 모델(튜닝 결과)로 **실시간 추론 엔드포인트** 생성
- 요청/응답 직렬화기(serializer/deserializer) 설정
- 배포된 엔드포인트로 **스모크 테스트**

> ⚠️ **비용 주의**: 실시간 엔드포인트는 삭제 전까지 **시간당 과금**됩니다.
> 워크샵이 끝나면 `08_cleanup.ipynb` 를 반드시 실행하세요.

`04_tuning.ipynb` 를 먼저 실행했어야 합니다.

## 0. 환경 복원 & 모델 재구성

튜닝에서 고른 최적 모델 아티팩트로 배포용 `Model` 객체를 만듭니다.

In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.model import Model

sess = sagemaker.Session()
role = get_execution_role()

%store -r bucket prefix xgb_image_uri best_model_data

model = Model(
    image_uri=xgb_image_uri,
    model_data=best_model_data,
    role=role,
    sagemaker_session=sess,
)
print("model artifact:", best_model_data)

## 1. 엔드포인트 배포

`model.deploy()` 는 내부적으로 **Model 등록 → EndpointConfig 생성 → Endpoint 생성**을 한 번에 처리합니다.
배포에는 수 분이 걸립니다. CSV로 요청/응답을 주고받도록 직렬화기를 지정합니다.

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer
import time, boto3

endpoint_name = "student-success-endpoint"

model.deploy(
    initial_instance_count=1,
    # TODO: 실시간 추론용 인스턴스 타입을 지정하세요 (예: "ml.m5.large").
    instance_type=____,
    endpoint_name=endpoint_name,
    wait=False,
)

sm_client = boto3.client("sagemaker")
print("deploying...", end="")
while True:
    status = sm_client.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    if status == "InService":
        print(" deployed!")
        break
    elif status == "Failed":
        print(" FAILED!")
        raise RuntimeError("Endpoint deployment failed")
    print(".", end="", flush=True)
    time.sleep(30)

# wait=False 시 predictor가 None이므로 직접 생성합니다.
from sagemaker.predictor import Predictor
predictor = Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sess,
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
)

## 2. 스모크 테스트

테스트셋에서 한 샘플을 꺼내 엔드포인트에 보내 봅니다.

> **스모크 테스트 vs A/B 테스트**
> - **스모크 테스트**: 배포 직후 "엔드포인트가 정상 응답하는지" 를 소수의 샘플로 빠르게 확인하는 것입니다.
>   기능적 정상 동작(응답 형식, 에러 없음, 합리적인 값)을 검증하는 **sanity check** 입니다.
> - **A/B 테스트**: 실제 트래픽의 일부를 새 모델로 보내고, 기존 모델과 **비즈니스 지표(전환율, 정확도 등)를 통계적으로 비교**하는 것입니다.
>   수일~수주간 실 사용자 트래픽으로 진행합니다.
>
> 즉, 스모크 테스트는 "작동하는가?"를, A/B 테스트는 "더 나은가?"를 답합니다.

In [ ]:
import numpy as np
import pandas as pd
from sagemaker.s3 import S3Downloader

%store -r test_x_s3
S3Downloader.download(test_x_s3, "smoke")
X = pd.read_csv("smoke/test_x.csv", header=None)
sample = X.iloc[0].tolist()

# CSVSerializer가 리스트를 CSV 한 줄로 변환해 전송합니다.
resp = predictor.predict(sample)          # -> [['0.1', '0.2', '0.7']]
proba = [float(p) for p in resp[0]]
classes = ["Dropout", "Enrolled", "Graduate"]

# TODO: 확률이 가장 큰 클래스를 예측 라벨로 고르세요. (힌트: int(np.argmax(proba)))
pred_idx = ____
print("probabilities:", dict(zip(classes, [round(p, 3) for p in proba])))
print("prediction   :", classes[pred_idx])

## 3. 엔드포인트 이름 저장

In [ ]:
%store endpoint_name
print("stored endpoint_name =", endpoint_name)

> 🔒 **보안 메모**: SageMaker 엔드포인트는 공개(public) URL이 아니라 **AWS IAM/SigV4 서명**으로만
> 호출할 수 있는 프라이빗 엔드포인트입니다. 뒤의 웹앱(`07`)도 IAM 자격증명으로 호출합니다.

✅ **완료!** 엔드포인트가 살아 있습니다. 다음은 `06_inference.ipynb` — 실시간 추론.